In [1]:
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
import random
import torch.optim as optim

### Seed + device

In [2]:
seed = 200

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
elif torch.backends.mps.is_available():
    torch.mps.manual_seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

### Sprawdzenie średniej i odchylenia dla każdego kanału

In [3]:
transform = transforms.Compose([transforms.ToTensor()])
dataset = torchvision.datasets.ImageFolder("train/", transform=transform)

map_class_to_idx = dataset.class_to_idx

all_samples = torch.stack([sample[0] for sample in dataset])
print(f"Data norm per channel: mean={all_samples.mean(axis=(0,2,3))} | std={all_samples.std(axis=(0,2,3))}")

Data norm per channel: mean=tensor([0.5204, 0.4950, 0.4381]) | std=tensor([0.2841, 0.2770, 0.2974])


### Transformacje na zbiorach

In [4]:
train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomCrop(64, 4),
        transforms.RandomRotation(16),
        transforms.ToTensor(),
        transforms.Normalize((0.5204, 0.4950, 0.4381), (0.2841, 0.2770, 0.2974))])

val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5204, 0.4950, 0.4381), (0.2841, 0.2770, 0.2974))])

train_dataset = torchvision.datasets.ImageFolder("train/", transform=train_transform)
val_dataset = torchvision.datasets.ImageFolder("train/", transform=val_transform)

dataset_size = len(train_dataset)
train_size = int(0.85 * dataset_size)
val_size = dataset_size - train_size

indices = torch.randperm(dataset_size)

train_subset = torch.utils.data.Subset(train_dataset, indices[:train_size])
val_subset = torch.utils.data.Subset(val_dataset, indices[train_size:])

print(f"Train: {len(train_subset)}")
print(f"Val: {len(val_subset)}")


Train: 74809
Val: 13202


### Dataloadery do treningu i walidacji

In [ ]:
train_loader = torch.utils.data.DataLoader(train_subset, batch_size=128, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_subset, batch_size=128, shuffle=False)

In [6]:
class Net(nn.Module):
    def __init__(self, n_classes):
        super().__init__()

        self.conv_layer1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.conv_layer2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.conv_layer3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.conv_layer4 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

    #     self.pool = nn.AdaptiveAvgPool2d((1, 1))

    #     self.dense_layer = nn.Sequential(
    #         nn.Dropout(0.3),
    #         nn.Linear(256, n_classes)
    #     )
    
    # def forward(self, x):
    #     x = self.conv_layer1(x)
    #     x = self.conv_layer2(x)
    #     x = self.conv_layer3(x)
    #     x = self.conv_layer4(x)
    #     x = self.pool(x)
    #     x = torch.flatten(x, 1)
    #     x = self.dense_layer(x)
    #     return x


        self.dense_layer = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, n_classes)
        )
    
    def forward(self, x):
        x = self.conv_layer1(x)
        x = self.conv_layer2(x)
        x = self.conv_layer3(x)
        x = self.conv_layer4(x)
        x = torch.flatten(x, 1)
        x = self.dense_layer(x)
        return x

In [7]:
EPOCHS = 50
model = Net(len(map_class_to_idx)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

In [8]:
def calc_accuracy(predictions: np.ndarray, targets: np.ndarray, n_classes=50):
    assert len(predictions) == len(targets)
    accuracies = []
    for i in range(n_classes):
        accuracies.append((predictions[targets == i] == i).sum() / (targets == i).sum())
    return np.mean(accuracies)

In [9]:
def train(model, train_loader, val_loader, criterion, optimizer, epochs):
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")

        model.train()
        train_loss = 0.0
        train_predictions = []
        train_targets = []

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            train_predictions.append(predicted.cpu())
            train_targets.append(labels.cpu())
        
        train_acc = calc_accuracy(torch.cat(train_predictions).numpy(), torch.cat(train_targets).numpy())
        train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        val_predictions = []
        val_targets = []

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
            
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                val_predictions.append(predicted.cpu())
                val_targets.append(labels.cpu())

        val_acc = calc_accuracy(torch.cat(val_predictions).numpy(), torch.cat(val_targets).numpy())
        val_loss = val_loss / len(val_loader)

        scheduler.step()

        print(f"Train loss: {train_loss:.3f}, accuracy: {train_acc*100:.2f}%")
        print(f"Val loss: {val_loss:.3f}, accuracy: {val_acc*100:.2f}%\n")


In [ ]:
train(model, train_loader, val_loader, criterion, optimizer, EPOCHS)

Epoch 1/50
Train loss: 3.228, accuracy: 14.49%
Val loss: 2.809, accuracy: 22.47%

Epoch 2/50
Train loss: 2.780, accuracy: 24.02%
Val loss: 2.456, accuracy: 30.99%

Epoch 3/50
Train loss: 2.497, accuracy: 30.93%
Val loss: 2.294, accuracy: 35.61%

Epoch 4/50
Train loss: 2.309, accuracy: 36.10%
Val loss: 2.064, accuracy: 41.32%

Epoch 5/50
Train loss: 2.151, accuracy: 40.18%
Val loss: 1.934, accuracy: 45.41%

Epoch 6/50
Train loss: 2.028, accuracy: 43.55%
Val loss: 1.911, accuracy: 46.45%

Epoch 7/50
Train loss: 1.915, accuracy: 46.57%
Val loss: 1.773, accuracy: 50.23%

Epoch 8/50
